In [0]:
from pyspark.sql.functions import col, count, countDistinct, sum, avg, round, year, month, desc, asc, when
from pyspark.sql.window import Window

# Criar o banco de dados da camada Gold
spark.sql("CREATE DATABASE IF NOT EXISTS workspace.gold")

DataFrame[]

In [0]:
# Carregando tabelas necessárias da Silver
df_pedido_total = spark.table("workspace.silver.fat_pedido_total")
df_itens = spark.table("workspace.silver.fat_itens_pedidos")
df_produtos = spark.table("workspace.silver.dim_produtos")

# Join para obter as categorias dos produtos
df_vendas = df_pedido_total.join(df_itens, "id_pedido") \
    .join(df_produtos, "id_produto")

# Agrupamento e métricas [cite: 108]
df_gold_comercial = df_vendas.groupBy(
    year("data_pedido").alias("ano_venda"),
    month("data_pedido").alias("mes_venda"),
    "categoria_produto"
).agg(
    countDistinct("id_pedido").alias("total_pedidos"),
    count("id_item").alias("qtd_itens_vendidos"),
    round(sum("valor_total_pago_brl"), 2).alias("receita_total_brl"),
    round(sum("valor_total_pago_usd"), 2).alias("receita_total_usd")
).withColumn("ticket_medio_brl", round(col("receita_total_brl") / col("total_pedidos"), 2))

# Gravação no modo overwrite [cite: 145]
df_gold_comercial.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.fat_vendas_comercial")

# Exibição dos Rankings Comerciais [cite: 110]
print("Top 5 Produtos MAIS Vendidos:")
display(df_vendas.groupBy("id_produto", "categoria_produto").agg(count("id_item").alias("qtd")).orderBy(desc("qtd")).limit(5))

print("Top 5 Produtos MENOS Vendidos:")
display(df_vendas.groupBy("id_produto", "categoria_produto").agg(count("id_item").alias("qtd")).orderBy(asc("qtd")).limit(5))

Top 5 Produtos MAIS Vendidos:


id_produto,categoria_produto,qtd
aca2eb7d00ea1a7b8ebd4e68314663af,moveis_decoracao,527
99a4788cb24856965c36a24e339b6058,cama_mesa_banho,488
422879e10f46682990de24d770e7f83d,ferramentas_jardim,484
389d119b48cf3043d311335e499d9c6b,ferramentas_jardim,392
368c6c730842d78016ad823897a372db,ferramentas_jardim,388


Top 5 Produtos MENOS Vendidos:


id_produto,categoria_produto,qtd
e9f2586707fb45ea0f9997c54f585842,esporte_lazer,1
e1c0a39b7f806727ea5121c60035ed3c,informatica_acessorios,1
fa11ecd35f999783e96ac500532d9d45,cool_stuff,1
8d7ab3701456fdbfe2526636ce15327a,malas_acessorios,1
cdb8d3c880b6639a70764c6734e6bb69,beleza_saude,1


In [0]:
# Carregando tabelas necessárias
df_avaliacoes = spark.table("workspace.silver.fat_avaliacoes_pedidos")
df_vendedores = spark.table("workspace.silver.dim_vendedores")

# Join para correlacionar avaliações, produtos e vendedores
df_qualidade = df_avaliacoes.join(df_itens, "id_pedido") \
    .join(df_produtos, "id_produto") \
    .join(df_vendedores, "id_vendedor")

# Agrupamento e métricas de satisfação [cite: 116, 120]
df_gold_satisfacao = df_qualidade.groupBy("categoria_produto", "id_vendedor", "estado").agg(
    count("id_avaliacao").alias("total_avaliacoes"),
    round(avg("nota_avaliacao"), 2).alias("avaliacao_media"),
    sum(when(col("nota_avaliacao") >= 4, 1).otherwise(0)).alias("total_avaliacoes_positivas"),
    sum(when(col("nota_avaliacao") <= 2, 1).otherwise(0)).alias("total_avaliacoes_negativas")
).withColumn("percentual_satisfacao", round((col("total_avaliacoes_positivas") / col("total_avaliacoes")) * 100, 2))

df_gold_satisfacao.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.fat_avaliacoes_clientes")

# Rankings de Qualidade com ordenação rigorosa [cite: 125]
# Primeiro por nota, depois por volume (desempate)
ranking_vendedores = df_gold_satisfacao.orderBy(desc("avaliacao_media"), desc("total_avaliacoes"))

print("Vendedor MAIS bem avaliado:")
display(ranking_vendedores.limit(1))

print("Vendedor MENOS bem avaliado:")
display(ranking_vendedores.orderBy(asc("avaliacao_media"), desc("total_avaliacoes")).limit(1))

Vendedor MAIS bem avaliado:


categoria_produto,id_vendedor,estado,total_avaliacoes,avaliacao_media,total_avaliacoes_positivas,total_avaliacoes_negativas,percentual_satisfacao
informatica_acessorios,a08692680c77d30a0b4280da5df01c5a,SP,17,5.0,17,0,100.0


Vendedor MENOS bem avaliado:


categoria_produto,id_vendedor,estado,total_avaliacoes,avaliacao_media,total_avaliacoes_positivas,total_avaliacoes_negativas,percentual_satisfacao
artes,a0e19590a0923cdd0614ea9427713ced,PR,7,1.0,0,7,0.0
